---
title: "Model Scene"
subtitle: "What can blurred exposures actually reveal?"
format:
  html:
    embed-resources: true
    html-math-method: mathml
    toc: true
    code-fold: true
    css: assets/scene.css
execute:
  echo: true
---

<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

A sharp-looking image can give a biased measurement. Here we reconstruct one
static modeled field from eight blurred exposures and ask whether the result
predicts four exposures that the solver never saw. We measure flux against
independent modeled truth, then deliberately break the PSF, noise and static-scene
assumptions. Every pixel comes from the analytic models distributed with this
example. No astronomical imagery, survey calibration, or downloaded catalog is used.

[Notebook source on GitHub](https://github.com/NVIDIA/cuPhoton/blob/main/examples/model-scene/model-scene.ipynb)

The full reference contains at least 50 independent noise realizations, with
identical draws shared between methods. A quick run exercises the APIs and
plots; its small sample supports no performance conclusion. All HTML selectors
choose saved results. They do not rerun a solver.

## A scene, a detector, and a unit

The <button class="scene-term" data-definition="The spatial light distribution before the exposure PSF blurs it." title="The spatial light distribution before the exposure PSF blurs it.">latent scene</button> contains two isolated stars, a close pair, and two extended Gaussian galaxies. Each source has a declared integrated flux, position and intrinsic width. A uniform background contributes two electrons per pixel per unit exposure. The stars have a small finite width of 0.22 pixel; their fine structure is deliberately undersampled.

<button class="scene-term" data-definition="A detector pixel records an integral over its area, rather than the value at its center." title="A detector pixel records an integral over its area, rather than the value at its center.">Pixel integration</button> matters when light changes appreciably within a pixel. Our physical renderer integrates continuous Gaussians over square pixels using normal-distribution CDF differences. Coordinates name pixel centers, and pixel edges lie half a pixel away. Flux units are electrons per unit exposure. Variance has the square of those units. All principal exposures have duration one, so counts and rates have the same numeric values.

<details class="science"><summary>Science Explanation: Continuous truth and discrete inference</summary>

For a Gaussian source of intrinsic width $s$ blurred by a Gaussian PSF of
width $p$, the continuous width is $\sqrt{s^2+p^2}$. The integral of a unit
one-dimensional Gaussian over pixel $j$ is

$$P_j=\Phi((j+1/2-\mu)/\sigma)-\Phi((j-1/2-\mu)/\sigma).$$

The two-dimensional integral is the product of the two axis integrals. Source
flux multiplies this unit integral. Finite-stamp losses remain visible; we do
not renormalize each source into the image. The renderer uses no xImage
convolution, torch operation or PSF helper. This independence exposes model
representation errors that a shared generator and solver could conceal.

</details>

In [ ]:
import json, os
from pathlib import Path
import numpy as np
from bokeh.io import output_notebook, show
from bokeh.resources import INLINE
from IPython.display import display, HTML
from cuphoton import __version__
from presentation import products, exposures, diagnostics, calibration_diagnostics
from scene import catalog
ROOT = Path(os.environ.get("CUPHOTON_MODEL_SCENE_OUTPUT", "results"))
STUDY = ROOT / "study"
output_notebook(resources=INLINE, hide_banner=True)
print("cuPhoton", __version__)
print("Profile:", json.loads((ROOT / "execution-profile.json").read_text()))
for source in catalog():
    print(source)

## What an exposure measures

<button class="scene-term" data-definition="The image of an ideal point source, including the blur represented by this model." title="The image of an ideal point source, including the blur represented by this model.">Point spread function (PSF)</button> describes how source light spreads across neighboring pixels. Twelve exposures have widths from 0.9 to 1.7 pixels and independent noise. They share a <button class="scene-term" title="A shared detector grid with no relative exposure shifts; this avoids resampling in the main experiment." data-definition="A shared detector grid with no relative exposure shifts; this avoids resampling in the main experiment.">zero-dither grid</button>. This experiment tests neither <button class="scene-term" title="Mapping between celestial coordinates and detector pixels. The zero-dither experiment does not validate this mapping." data-definition="Mapping between celestial coordinates and detector pixels. The zero-dither experiment does not validate this mapping.">WCS alignment</button> nor <button class="scene-term" title="Estimating values on a new pixel grid, which can introduce correlations between neighboring noise samples." data-definition="Estimating values on a new pixel grid, which can introduce correlations between neighboring noise samples.">resampling</button>.

In the main case, <button class="scene-term" title="Independent normal-distribution noise with specified mean and variance; the main case uses a standard deviation of two electrons per pixel." data-definition="Independent normal-distribution noise with specified mean and variance; the main case uses a standard deviation of two electrons per pixel.">Gaussian noise</button> has exactly known <button class="scene-term" title="Expected squared deviation from the mean. Its units are the square of the intensity units." data-definition="Expected squared deviation from the mean. Its units are the square of the intensity units.">variance</button> four. The physical generator performs continuous convolution before pixel integration. xImage applies finite, sampled kernels to a discrete latent image. Those operators differ slightly, especially for undersampled stars. The matched-operator control isolates that discrepancy by generating observations with a separate SciPy discrete convolution.

<details class="science"><summary>Science Explanation: The forward model</summary>

For exposure $e$ and pixel $i$, the inference model is

$$y_{ei}=(H_e x)_i+n_{ei},\qquad n_{ei}\sim\mathcal N(0,v_{ei}).$$

Here $x$ includes the known constant background, $H_e$ denotes the supplied
fixed PSF convolution, and the independent variances $v_{ei}$ are positive.
A symmetric Gaussian PSF simplifies the teaching field; separate asymmetric
operator controls are needed to detect a kernel flip or transpose. Four
held-out frames test a different blur of the same scene. Their noise is
independent, although their <button class="scene-term" title="Uncertainty in the forward prediction caused by estimating a scene from noisy fitting data." data-definition="Uncertainty in the forward prediction caused by estimating a scene from noisy fitting data.">prediction uncertainty</button> comes from the fitted image.

</details>

## Read the array contract before fitting

Images, variances, masks and background use `(batch, y, x, frame)`. PSFs use `(kernel_y, kernel_x, channel, frame)`. The singleton batch and channel axes have different meanings. A transpose that preserves array rank can still change the science.

Masks are boolean: **True means good**. The outlier generator records which pixels it contaminated, and the explicit-mask investigation uses that record. This is an idealized quality flag, not a claim that a detector would identify all outliers. The example rejects nonfinite or nonpositive variances before starting a fit. Exposure <button class="scene-term" title="Rescaling a PSF so its total discrete weight has a declared value, usually one for flux conservation." data-definition="Rescaling a PSF so its total discrete weight has a declared value, usually one for flux conservation.">normalization</button> divides intensity by time and variance by time squared, exactly once. Counts-versus-rates controls exercise the actual workflow preprocessing.

In [ ]:
from cuphoton.ximage.data import load_ximage_bundle, validate_bundle
with load_ximage_bundle(STUDY / "trial-000/l2-bundle") as bundle:
    print("Bundle validation:", validate_bundle(bundle))
    print("Images:", bundle.images.shape, "PSFs:", bundle.psfs.shape)
    print("Good-mask dtype:", bundle.masks.dtype)
    assert np.all(np.isfinite(bundle.variances))
    assert np.all(bundle.variances > 0)

## Build the ordinary coadd

<button class="scene-term" data-definition="A combination of several exposures, here their arithmetic mean after duration normalization." title="A combination of several exposures, here their arithmetic mean after duration normalization.">Coadd</button> reduces independent noise. It also mixes the exposure PSFs. Averaging eight images does not by itself establish finer spatial resolution. Equal Gaussian variances make equal weights appropriate for this experiment. The image-level comparison first blurs each constituent to a common attainable Gaussian width of 2.4 pixels, then averages. Reconstruction receives the corresponding additional blur. This comparison spends resolution to make flux measurements comparable. Discrete smoothing approximates continuous blur, and the noiseless physical control measures the residual <button class="scene-term" title="The average signed measurement error over independent realizations of the declared noise model." data-definition="The average signed measurement error over independent realizations of the declared noise model.">bias</button> floor.

<details class="science"><summary>Science Explanation: Why the target resolution is broader</summary>

A Gaussian blur adds variances. To turn a width $p_e$ exposure into a
width $q$ target, the additional blur has width $\sqrt{q^2-p_e^2}$, which
requires $q\ge p_e$. We choose $q=2.4$ before evaluating the test scene.
A coadd PSF is a mixture of the exposure PSFs; it generally has no single
Gaussian width. Blurring each constituent separately avoids replacing that
mixture with an unjustified single Gaussian. The source's intrinsic width
remains part of the target image.

</details>

## Reconstruct with xImage L2

<button class="scene-term" data-definition="Inference of one nonnegative static scene whose supplied-PSF predictions agree with several exposures." title="Inference of one nonnegative static scene whose supplied-PSF predictions agree with several exposures.">Reconstruction</button> uses eight frames and fixed kernels. It solves no PSF parameters. The visible call below repeats the first saved fit with the same recorded configuration. The full study performs paired L2 and Huber fits on each noise realization.

<details class="science"><summary>Science Explanation: The weighted objective</summary>

The L2 objective is proportional to

$$L(x)=\frac12\sum_{e,i}m_{ei}\frac{[y_{ei}-(H_ex)_i]^2}{v_{ei}},\qquad x\ge0.$$

The mask $m$ is one for valid pixels. Pixels with larger variance receive
less weight. Multiplicative updates preserve <button class="scene-term" title="The constraint that inferred scene intensity cannot be negative; this can bias measurements in low-signal regions." data-definition="The constraint that inferred scene intensity cannot be negative; this can bias measurements in low-signal regions.">nonnegativity</button> under their
supported positive-kernel assumptions. An update stopping score is not a
<button class="scene-term" title="An interval constructed by a repeated-sampling procedure with a specified coverage probability under its assumptions." data-definition="An interval constructed by a repeated-sampling procedure with a specified coverage probability under its assumptions.">confidence interval</button>. Iteration count regularizes this finite inverse problem;
the reference fixes 300 iterations, a constant Huber threshold, and a very
small stopping tolerance after a separate development-scene sweep.

</details>

In [ ]:
import yaml
from cuphoton.ximage.workflows import run_reconstruction
config = yaml.safe_load((STUDY / "trial-000/l2.yaml").read_text())
config["output"]["name"] = "notebook-l2"
config_path = ROOT / "notebook-l2.yaml"
config_path.write_text(yaml.safe_dump(config))
l2_run = run_reconstruction(config_path)
print(l2_run.summary["trace_summary"])

## Compare robust losses

<button class="scene-term" data-definition="A loss that is quadratic near zero standardized residual and linear beyond a fixed threshold." title="A loss that is quadratic near zero standardized residual and linear beyond a fixed threshold.">Huber loss</button> limits the influence of large <button class="scene-term" title="Observed minus predicted intensity divided by the stated pixel standard deviation." data-definition="Observed minus predicted intensity divided by the stated pixel standard deviation.">standardized residuals</button>. It can help when a small fraction of pixels have large positive artifacts. Robustness also changes the response to ordinary noise and model mismatch. It need not improve every source or metric. Our threshold is two standardized residual units throughout the fit. A changing threshold would make the loss trace a sequence of different objectives.

<details class="science"><summary>Science Explanation: Residual scale and weighting</summary>

For $r=(y-Hx)/\sqrt v$, the Huber loss is

$$\rho_\delta(r)=\begin{cases}r^2/2,&|r|\le\delta,\\
\delta(|r|-\delta/2),&|r|>\delta.\end{cases}$$

Scaling all variances by four divides standardized residuals by two. It changes
the effective Huber transition in the original intensity units. The numerical
control compares this with doubling the threshold. L2's exact minimizer is
unchanged by a global variance factor, while its reported objective rescales.
Finite arithmetic and weakly constrained halo pixels can break exact numeric
invariance; the controls record that boundary sensitivity.

</details>

In [ ]:
show(products(STUDY, {"Principal Gaussian case": "trial-000",
    "Outliers": "outliers", "Explicit quality mask": "explicit-mask"}))

## Keep real border context

<button class="scene-term" data-definition="Convolution evaluated where its kernel has real surrounding image support, without inventing a zero-valued measurement border." title="Convolution evaluated where its kernel has real surrounding image support, without inventing a zero-valued measurement border.">VALID padding</button> uses a 15×15 kernel, a 48×48 measured ROI, and surrounding pixels from the generated 72×72 field. The saved reconstruction is 62×62 because it includes a seven-pixel halo. Forward convolution of that image yields the measured 48×48 interior. The outer latent corners are weakly constrained. We retain their numerical instability in the controls and exclude them from catalog measurements. Apertures must lie at least five pixels from the measurement boundary. No zero-filled edge is interpreted as a detector observation.

## Calibrate a PSF from independent star data

<button class="scene-term" data-definition="Estimating the blur from separate calibration observations before reconstructing a scene." title="Estimating the blur from separate calibration observations before reconstructing a scene.">PSF calibration</button> has two distinct steps here. First, an example-authored estimator subtracts the known background from centered calibration stars, clips negative noisy samples, and normalizes the stamp. Each idealized on-grid star supplies 100,000 electrons and independent Gaussian read noise. Its known centering avoids a shift-estimation problem. Second, `run_psf_solve` factors that measured stamp through a Gaussian reference. The public solver receives PSF images; it does not locate or fit stars.

<details class="science"><summary>Science Explanation: The Gaussian reference changes the target</summary>

PSF factorization approximates $f_e=g*h_e$, with reference width
$\sigma_g=0.55$ pixel. Reconstructing with $h_e$ targets $g*x$, rather than
the unconvolved scene $x$. The common-resolution comparison therefore adds
only $\sqrt{2.4^2-0.55^2}$ pixels of Gaussian blur to that product.
Finite <button class="scene-term" title="Random measurement error in the separate observations used to estimate PSFs." data-definition="Random measurement error in the separate observations used to estimate PSFs.">calibration noise</button>, stamp clipping and imperfect factorization each
contribute error. Perfect-known-kernel and independently repeated calibration
draws separate those effects. A calibration error shared by several science
realizations is a common <button class="scene-term" title="An error shared across measurements, such as a fixed PSF calibration error reused for many science exposures." data-definition="An error shared across measurements, such as a fixed PSF calibration error reused for many science exposures.">systematic</button>, not additional independent science noise.

</details>

In [ ]:
from cuphoton.ximage.workflows import run_psf_solve
psf_config = yaml.safe_load((STUDY / "calibration/calibration-mm.yaml").read_text())
psf_config["output"]["name"] = "notebook-psf-mm"
psf_path = ROOT / "notebook-psf-mm.yaml"
psf_path.write_text(yaml.safe_dump(psf_config))
psf_run = run_psf_solve(psf_path)
print(psf_run.summary["method"], psf_run.summary["psfs_shape"])
show(calibration_diagnostics(STUDY))

## Inspect the saved calibration kernels

Multiplicative PSF updates are the main investigation. Adam supplies an additional optimizer comparison. Adam can return negative kernel elements. The reference records its minimum, normalization and prediction error; such a kernel is excluded from multiplicative scene reconstruction when it violates positive-kernel assumptions. A small optimization loss alone does not establish a physically admissible calibrated kernel. The finite-calibration experiment repeats independent calibration draws on the same noiseless science scene. Its scatter describes the chosen calibration model at the declared star flux. It does not establish survey calibration quality.

## Reveal truth at a fair resolution

The truth image below comes from the independent physical model. All four panels target the same 2.4-pixel Gaussian resolution. Pan or zoom one panel to move the others. Tap a pixel to inspect its saved numeric value. The exact pixel values remain available independently of the display range. Use the noiseless and matched-operator cases to distinguish observational noise from <button class="scene-term" title="Representing continuous light distributions with a finite pixel grid and finite numerical kernels." data-definition="Representing continuous light distributions with a finite pixel grid and finite numerical kernels.">discretization</button> or finite-iteration bias.

In [ ]:
show(products(STUDY, {"Physical renderer": "trial-000",
    "Noiseless physical control": "noiseless-physical",
    "Matched discrete operator": "matched-operator",
    "Calibrated MM kernels": "calibrated-mm",
    "Perfect kernels through factorization": "perfect-factorization"}))

## Declare the flux endpoint

<button class="scene-term" data-definition="Background-subtracted light summed inside a declared circular region, including any neighboring light inside that region." title="Background-subtracted light summed inside a declared circular region, including any neighboring light inside that region.">Aperture flux</button> is the primary endpoint: radius three pixels around each fixed catalog position, after matching resolution. We compare each aperture with the identical aperture on known target truth. These are local light measurements. They include neighbors and are not deblended total source fluxes. <button class="scene-term" title="Measured quantity minus known target quantity, divided by that known target quantity." data-definition="Measured quantity minus known target quantity, divided by that known target quantity.">Fractional error</button> is `(measured − truth) / truth`. We report every fixed source by input brightness and environment, plus class summaries. The catalog is small and fixed, so the experiment estimates noise sensitivity for these sources; it does not estimate a population-wide brightness or <button class="scene-term" title="Overlap of light from nearby sources, which can bias source-by-source measurements." data-definition="Overlap of light from nearby sources, which can bias source-by-source measurements.">crowding</button> law.

<details class="science"><summary>Science Explanation: Paired uncertainty and detectable bias</summary>

Each independent noise realization supplies one result per method. We form
method differences within the same realization, preserving the <button class="scene-term" title="Expected product of two centered errors. Paired methods share input noise and therefore have correlated errors." data-definition="Expected product of two centered errors. Paired methods share input noise and therefore have correlated errors.">covariance</button> from
shared input noise. We average sources within a class before estimating a
class <button class="scene-term" title="Estimated standard deviation of an estimator, here the mean error across independent noise realizations." data-definition="Estimated standard deviation of an estimator, here the mean error across independent noise realizations.">standard error</button> across realizations. Sources in one image are not counted
as independent noise realizations.

For $N$ independent trials, the estimated standard error of mean bias is
$s/\sqrt N$. The page shows approximate 95% mean-bias intervals and reports
$2.8s/\sqrt N$ as an approximate detectable bias at <button class="scene-term" title="An approximate 80% probability of detecting the stated nonzero bias with the chosen test and sampling assumptions." data-definition="An approximate 80% probability of detecting the stated nonzero bias with the chosen test and sampling assumptions.">80% power</button> for a two-sided
5% test. Scatter estimates at $N=50$ remain relatively imprecise. No broad
method ranking follows from a difference smaller than its measured precision.

</details>

In [ ]:
summary = json.loads((STUDY / "summary.json").read_text())
print(json.dumps(summary, indent=2))
show(diagnostics(STUDY))

## Positions, blends and extended light

<button class="scene-term" data-definition="The intensity-weighted mean position inside the stated measurement aperture." title="The intensity-weighted mean position inside the stated measurement aperture.">Centroid</button> shifts when neighboring light enters an aperture. We compare measured centroids with target-truth aperture centroids. For the close pair we also record the distance between the two overlapping-aperture centroids and compare it with the intrinsic three-pixel separation. This deliberately simple estimator is biased by blending. A clean-looking pair cannot establish an unbiased deblended separation measurement. <button class="scene-term" title="Measured within a ring between two specified radii around a source position." data-definition="Measured within a ring between two specified radii around a source position.">Annular</button> mean-light profiles retain the extended-source evidence that a single <button class="scene-term" title="A measure of fine-scale image structure that can increase from either recovered detail or amplified noise." data-definition="A measure of fine-scale image structure that can increase from either recovered detail or amplified noise.">sharpness</button> number discards. The complete numerical artifacts include per-source profiles and centroid errors for every realization.

## Predict exposures the solver never saw

<button class="scene-term" data-definition="A forward model evaluated on an exposure excluded from fitting, using its supplied PSF." title="A forward model evaluated on an exposure excluded from fitting, using its supplied PSF.">Held-out prediction</button> tests whether one reconstructed scene explains different blurs. Select a frame below. Frames 1–8 were fitted; frames 9–12 were held out. Residual means and spatial patterns matter alongside their magnitude. A residual second moment divided by known pixel variance is a diagnostic, not an automatic chi-square test with mean one. Fitting, positivity, spatial correlations and uncertainty in the predicted image change its interpretation. An independently noisy held-out frame removes one source of circularity, but it does not remove prediction uncertainty.

In [ ]:
show(exposures(STUDY))

## Make the PSF wrong by five percent

The next selector supplies kernels that are uniformly 5% too narrow or 5% too broad. Every exposure shares the sign of that error, including held-out kernels. A common PSF error can be absorbed partly into the inferred scene and evade a residual-only check. Truth-based aperture, centroid and profile errors quantify its measurement consequences. Small apertures and native-resolution profile fits can be more sensitive than common-resolution <button class="scene-term" title="The sum of background-subtracted light inside a specified region, including light from any neighbor inside that region." data-definition="The sum of background-subtracted light inside a specified region, including light from any neighbor inside that region.">aperture flux</button>. A weak flux effect is retained as a finite-sensitivity result.

In [ ]:
show(products(STUDY, {"Supplied true kernels": "trial-000",
    "PSF width −5%": "psf-width-0.95", "PSF width +5%": "psf-width-1.05"}))

## Separate outliers, variances and photon noise

The outlier case adds 100 electrons to roughly 0.3% of pixels. L2, Huber and an ideal explicit mask receive the same realization. Masking removes known contaminated measurements; Huber reduces influence according to residual size. They use different information, so the mask comparison is an idealized control.

Multiplying variances by four changes the interpretation of standardized residuals and Huber weighting. The photon/read-noise case instead draws <button class="scene-term" title="A count distribution whose variance equals its mean; used in the separately labeled photon-noise stress case." data-definition="A count distribution whose variance equals its mean; used in the separately labeled photon-noise stress case.">Poisson</button> electrons and adds Gaussian read noise. Its known variance equals expectation plus read variance. That likelihood differs from the exactly Gaussian principal case even though its first two moments are supplied. The page preserves the distinction when reporting results.

In [ ]:
show(products(STUDY, {"Known Gaussian variance": "trial-000",
    "Variance multiplied by four": "variance-times-four",
    "Photon plus read noise": "photon-read"}))
from presentation import case_measurement_table
display(HTML(case_measurement_table(STUDY)))


## Break the static-scene assumption

<button class="scene-term" data-definition="A source whose flux changes between exposures, violating the single static scene assumed by this fit." title="A source whose flux changes between exposures, violating the single static scene assumed by this fit.">Transient</button> appears in a minority of frames at a new position. A static fit can dilute its flux across the compromise image, redistribute light, or leave inconsistent frame residuals. The transient is present in fitted frames 2 and 7 and held-out frame 10. Select those frames and compare them with their neighbors. A robust loss may suppress variability that a time-domain analysis would want to preserve. This experiment has no classifier or alert stream; the evidence is the inconsistency of individual image predictions.

In [ ]:
show(products(STUDY, {"Static source model": "trial-000", "Transient input": "transient"}))
show(exposures(STUDY, "transient"))

## What the native metrics can establish

The native evaluator records coadd-relative <button class="scene-term" title="Peak signal-to-noise ratio, an image-difference metric whose interpretation depends on the reference image and intensity range." data-definition="Peak signal-to-noise ratio, an image-difference metric whose interpretation depends on the reference image and intensity range.">PSNR</button>, <button class="scene-term" title="Structural similarity, an image-comparison diagnostic that does not by itself measure truth recovery." data-definition="Structural similarity, an image-comparison diagnostic that does not by itself measure truth recovery.">SSIM</button> and sharpness. These diagnostics share input information and compare different effective resolutions. They describe image differences; they cannot establish reconstruction accuracy. The primary truth-based endpoint, independent calibration controls and individual held-out predictions supply the scientific qualification. The zero-dither experiment supports no claim about shift recovery, resampling covariance, WCS alignment or arbitrary super-resolution.

In [ ]:
from cuphoton.ximage.evaluation import evaluate_reconstruction_run
secondary = evaluate_reconstruction_run(l2_run.run_dir, metrics_backend="cpu")
print(json.dumps(secondary, indent=2, default=str))

## Reproduce and inspect the evidence

The execution receipt records source hashes, protocol hash, seeds, dependency versions and artifact hashes. The full numerical study stays outside Git. The page embeds selected first-realization arrays and complete summary diagnostics, keeping repeated-trial arrays out of the HTML. Every selector is a precomputed experiment selector. Reading the exported page requires no Python process, kernel, WebSocket, API, CDN or network connection.

See `README.md` for clean-environment execution and `PUBLISHING.md` for static packaging and the remaining versioned-Pages integration work. Opening or merging the source PR does not publish this chapter.

The reconstruction follows the ImageMM approach: Yashil Sukurdeep, Tamás Budavári, Andrew J. Connolly and Fausto Navarro, [ImageMM: Joint Multi-frame Image Restoration and Super-resolution](https://doi.org/10.3847/1538-3881/adfb72), *The Astronomical Journal* 170, 233 (2025). Cite that paper and the cuPhoton version printed by this execution. The declared teaching models and source catalog are original to this example.